# Neural Network Deployment — From PyTorch to Production
## Real Autoware Models · Real Waymo Data · Real Benchmarks

This notebook uses **SceneSeg** — Autoware's production segmentation model — and real **Waymo driving frames**.

| Test | Runtime | Device |
|------|---------|--------|
| Test 1 | PyTorch | CPU (baseline) |
| Test 2 | PyTorch | GPU |
| Test 3 | ONNX Runtime | GPU |
| Test 4 | TensorRT FP16 | GPU |

> **Requirements:** Google Colab · T4 GPU runtime

---
## 0. Setup

In [ ]:
!pip install torch torchvision onnx onnxsim onnxruntime-gpu gdown tensorrt pycuda -q
import warnings; warnings.filterwarnings('ignore')
print("✓ Packages ready")

In [ ]:
import torch
import torchvision.transforms as T
import numpy as np
import time, os, glob
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2, onnx, onnxsim
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType

print(f"PyTorch   {torch.__version__}")
print(f"ORT       {ort.__version__}")
print(f"CUDA      {torch.cuda.is_available()}  —  " +
      (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"))

---
## 1. Download Data & Models

Real **Waymo driving frames** + Autoware's **SceneSeg** model at multiple precisions.

In [ ]:
!mkdir -p /content/data /content/models

# Real Waymo driving frames
!wget -qq https://optical-flow-data.s3.eu-west-3.amazonaws.com/waymo_images.zip \
     -O /content/data/waymo.zip
!unzip -qq /content/data/waymo.zip -d /content/data/

# SceneSeg ONNX models (pre-built by Autoware at multiple precisions)
!gdown -O /content/models/ 'https://docs.google.com/uc?export=download&id=1l-dniunvYyFKvLD7k16Png3AsVTuMl9f'
!gdown -O /content/models/ 'https://docs.google.com/uc?export=download&id=19gMPt_1z4eujo4jm5XKuH-8eafh-wJC6'
!gdown -O /content/models/ 'https://docs.google.com/uc?export=download&id=1zCworKw4aQ9_hDBkHfj1-sXitAAebl5Y'

# SceneSeg Traced PyTorch model (TorchScript — no class definition needed)
!gdown '1G2pKrjEGLGY1ouQdNPh11N-5LlmDI7ES' -O /content/models/SceneSeg_traced.pt

print("\n📁 Models:")
for f in sorted(glob.glob('/content/models/*')):
    print(f"  {Path(f).name:<40}  {os.path.getsize(f)/1024/1024:.1f} MB")

frames = sorted(
    glob.glob('/content/data/**/*.jpg', recursive=True) +
    glob.glob('/content/data/**/*.png', recursive=True)
)
print(f"\n📷 Waymo frames: {len(frames)} images")

---
## 2. Load SceneSeg (PyTorch) → Visualize → Benchmark

### What is SceneSeg?
SceneSeg is Autoware's semantic segmentation model. It labels every pixel as one of **3 classes**:
- 🔴 **Background** — sky, buildings, vegetation
- 🟣 **Foreground objects** — cars, pedestrians, cyclists
- 🟢 **Drivable road** — the surface the AV drives on

It runs on every camera frame and feeds directly into the AV's planning module.

### What is a Traced PyTorch model?
A **Traced model** is created with `torch.jit.trace()`. PyTorch records every operation as it runs
on a sample input, freezing the computation graph as **TorchScript**. This means:
- No Python runtime needed at deployment
- Loads with `torch.jit.load()` — **no class definition required**
- Exports to ONNX cleanly
- What Autoware ships to C++ inference nodes

In [ ]:
# Load the traced SceneSeg model — no class definition needed
model = torch.jit.load('/content/models/SceneSeg_traced.pt', map_location='cpu')
model.eval()
print("✓ SceneSeg traced model loaded")

H, W = 256, 512   # will be overridden below by auto-probe

# Probe the exact input size the model was traced with.
# Traced (TorchScript) models bake in tensor shapes — we must use the
# exact same H×W that was used during torch.jit.trace().
CANDIDATE_SIZES = [(256, 512), (320, 576), (320, 640), (384, 640), (480, 640), (512, 512)]

with torch.no_grad():
    for h, w in CANDIDATE_SIZES:
        try:
            model(torch.zeros(1, 3, h, w))
            H, W = h, w
            print(f'✓ Working input size found: H={H}  W={W}')
            break
        except Exception as e:
            print(f'  ({h:3d},{w:3d}) — {str(e)[:70]}')
    else:
        raise RuntimeError('No candidate size worked. Add more sizes to CANDIDATE_SIZES.')

preprocess = T.Compose([
    T.Resize((H, W)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# SceneSeg 3-class colour palette (from Autoware workshop)
COLORS = np.array([
    [255,  93,  61],  # 0 = background
    [145,  28, 255],  # 1 = foreground objects
    [220, 255,   0],  # 2 = drivable road
], dtype=np.uint8)

def scene_vis(orig_bgr, raw_out):
    """Blend segmentation colours onto the original frame."""
    a = np.array(raw_out)
    if a.ndim == 4:
        ch = a.shape[1]
        class_map = np.argmax(a[0], axis=0) if ch > 1 else (a[0, 0] > 0.0).astype(np.int32)
    elif a.ndim == 3:
        class_map = np.argmax(a, axis=0) if a.shape[0] > 1 else (a[0] > 0.0).astype(np.int32)
    else:
        class_map = (a.squeeze() > 0.0).astype(np.int32)
    class_map = np.clip(class_map, 0, 2)
    vis_col = COLORS[class_map.astype(int)]
    if vis_col.shape[:2] != orig_bgr.shape[:2]:
        vis_col = cv2.resize(vis_col, (orig_bgr.shape[1], orig_bgr.shape[0]),
                             interpolation=cv2.INTER_NEAREST)
    return cv2.addWeighted(orig_bgr, 0.6, vis_col, 0.4, 0)

print("✓ Preprocessing + scene_vis ready")

In [ ]:
# Load Waymo frames and show them
frames = sorted(
    glob.glob('/content/data/**/*.jpg', recursive=True) +
    glob.glob('/content/data/**/*.png', recursive=True)
)
show = frames[:6]

fig, axes = plt.subplots(1, len(show), figsize=(20, 3))
for ax, f in zip(axes, show):
    ax.imshow(cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB))
    ax.set_title(Path(f).name[:18], fontsize=7); ax.axis('off')
plt.suptitle('Waymo Driving Frames', fontsize=12)
plt.tight_layout(); plt.show()

# Pick the first frame as our test image
frame_bgr    = cv2.imread(frames[0])
frame_rgb    = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
pil_frame    = Image.fromarray(frame_rgb)
input_tensor = preprocess(pil_frame).unsqueeze(0)   # (1,3,H,W) CPU
print(f"Input tensor: {input_tensor.shape}")

In [ ]:
# Run PyTorch inference and visualise
with torch.no_grad():
    raw_out = model(input_tensor)

# TorchScript models may return a tensor or a list/tuple
if isinstance(raw_out, (list, tuple)):
    raw_out = raw_out[0]
seg_np = raw_out.cpu().numpy()   # keep for ORT comparison later

overlay = scene_vis(cv2.resize(frame_bgr, (W, H)), seg_np)

# Build colour legend
legend = [mpatches.Patch(color=np.array(c)/255, label=lbl)
          for c, lbl in zip([[255,93,61],[145,28,255],[220,255,0]],
                             ['Background','Foreground','Road'])]

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].imshow(cv2.resize(frame_rgb, (W, H)));          axes[0].set_title('Input Frame');        axes[0].axis('off')
axes[1].imshow(COLORS[np.clip(np.argmax(seg_np[0],0) if seg_np.shape[1]>1 else (seg_np[0,0]>0).astype(int), 0, 2)]);
axes[1].set_title('Segmentation Mask'); axes[1].axis('off'); axes[1].legend(handles=legend, loc='lower right', fontsize=8)
axes[2].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)); axes[2].set_title('Overlay'); axes[2].axis('off')
plt.suptitle('SceneSeg — Autoware Semantic Segmentation (3 classes)', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# Benchmark helper — returns the full time distribution for rich plots later
# n_runs is intentionally lower for CPU (20) vs GPU (100) — CPU passes are slow
def benchmark(run_fn, name, n_warmup=5, n_runs=50, use_cuda=False):
    with torch.no_grad():
        for _ in range(n_warmup):
            run_fn()
    if use_cuda:
        torch.cuda.synchronize()
    times = []
    with torch.no_grad():
        for _ in range(n_runs):
            t0 = time.perf_counter()
            run_fn()
            if use_cuda:
                torch.cuda.synchronize()
            times.append((time.perf_counter() - t0) * 1000)
    t = np.array(times)
    p50, p95 = np.percentile(t, 50), np.percentile(t, 95)
    print(f'{name:.<58} avg={t.mean():6.1f}ms  p50={p50:5.1f}ms  p95={p95:5.1f}ms  ({1000/t.mean():.0f} FPS)')
    return t

# ── Test 1: PyTorch CPU ───────────────────────────────────────────────────────
# SceneSeg is a GPU model — CPU is slow by design. We time a single pass
# just to establish a baseline; no point waiting 5+ minutes for statistics.
with torch.no_grad():
    model(input_tensor)   # one warmup
    t0 = time.perf_counter()
    model(input_tensor)
    cpu_ms = (time.perf_counter() - t0) * 1000

pytorch_cpu_times = np.array([cpu_ms])   # single-element array keeps downstream plots working
print(f'Test 1 · PyTorch CPU{"."*37} {cpu_ms:.1f} ms  ({1000/cpu_ms:.1f} FPS)  [single run — CPU is reference only]')

In [ ]:
# ── Test 2: PyTorch GPU ──────────────────────────────────────────────────────
if not torch.cuda.is_available():
    print('⚠ No GPU — skipping Test 2'); pytorch_gpu_times = None
else:
    model_gpu  = model.cuda()
    input_gpu  = input_tensor.cuda()
    pytorch_gpu_times = benchmark(
        lambda: model_gpu(input_gpu),
        f'Test 2 · PyTorch GPU ({torch.cuda.get_device_name(0)})',
        use_cuda=True,
    )
    print(f'\nGPU speedup over CPU: {pytorch_cpu_times.mean()/pytorch_gpu_times.mean():.1f}x')

---
## 3. Export to ONNX → Run with ONNX Runtime

The traced (TorchScript) model exports to ONNX without any wrapper. Once you have an ONNX file,
loading it for inference is **6 lines** — this is what production deployments look like.

In [ ]:
# Export the traced SceneSeg model to ONNX
# TorchScript traces export cleanly — no wrapper class needed
ONNX_EXPORT_PATH = '/content/models/SceneSeg_export.onnx'
img_np = input_tensor.numpy()   # CPU numpy — stays CPU throughout ONNX path

torch.onnx.export(
    model, input_tensor, ONNX_EXPORT_PATH,
    input_names=['image'], output_names=['segmentation'],
    opset_version=17,
)

onnx_model = onnx.load(ONNX_EXPORT_PATH)
onnx.checker.check_model(onnx_model)
simplified, ok = onnxsim.simplify(onnx_model)
if ok:
    onnx.save(simplified, ONNX_EXPORT_PATH)

size_mb = os.path.getsize(ONNX_EXPORT_PATH) / 1024 / 1024
print(f'✓ Exported  →  {ONNX_EXPORT_PATH}  ({size_mb:.1f} MB)')

In [ ]:
# ── Load with ONNX Runtime — production-style, minimal ───────────────────────
# Use the pre-built Autoware FP32 ONNX if available, else our export
onnx_files = sorted(glob.glob('/content/models/*.onnx'))
MODEL_FILE  = Path(next((f for f in onnx_files if 'FP32' in f), ONNX_EXPORT_PATH))
print(f'Using: {MODEL_FILE.name}')

providers = (
    ['CUDAExecutionProvider', 'CPUExecutionProvider']
    if 'CUDAExecutionProvider' in ort.get_available_providers()
    else ['CPUExecutionProvider']
)
sess     = ort.InferenceSession(MODEL_FILE, providers=providers)
inp_meta = sess.get_inputs()[0]

try:
    H_ort = int(inp_meta.shape[2]) if inp_meta.shape[2] is not None else H
    W_ort = int(inp_meta.shape[3]) if inp_meta.shape[3] is not None else W
except Exception:
    H_ort, W_ort = H, W

active = sess.get_providers()[0]
print(f'Provider : {active}')
print(f'Input    : {inp_meta.name}  {inp_meta.shape}')

In [ ]:
# ORT inference — pass CPU numpy; CUDA provider handles H2D internally
ort_out = sess.run(None, {inp_meta.name: img_np})[0]

ort_overlay = scene_vis(cv2.resize(frame_bgr, (W_ort, H_ort)), ort_out)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].imshow(cv2.cvtColor(overlay,     cv2.COLOR_BGR2RGB)); axes[0].set_title('PyTorch output');            axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(ort_overlay, cv2.COLOR_BGR2RGB)); axes[1].set_title(f'ONNX Runtime ({active})'); axes[1].axis('off')
plt.suptitle('Equivalence check: PyTorch vs ONNX Runtime', fontsize=13)
plt.tight_layout(); plt.show()

max_diff = np.abs(seg_np - ort_out).max()
print(f'Max numerical difference: {max_diff:.6f}  (< 0.01 ✓)')

In [ ]:
# ── Test 3: ONNX Runtime GPU ─────────────────────────────────────────────────
onnx_gpu_times = benchmark(
    lambda: sess.run(None, {inp_meta.name: img_np}),
    f'Test 3 · ONNX Runtime ({active})',
    use_cuda=('CUDA' in active),
)
if pytorch_gpu_times is not None:
    print(f'ORT vs PyTorch GPU: {pytorch_gpu_times.mean()/onnx_gpu_times.mean():.2f}x')

---
## 4. Precision: FP32 vs FP16 vs INT8

In production you never deploy in FP32 if you don't have to.
Autoware ships models at multiple precisions — let's compare them all.

In [ ]:
# Dynamic INT8 quantization from the FP32 ONNX
QUANT_PATH = '/content/models/SceneSeg_INT8_ort.onnx'
quantize_dynamic(str(MODEL_FILE), QUANT_PATH, weight_type=QuantType.QInt8)

orig_mb  = os.path.getsize(MODEL_FILE) / 1024 / 1024
quant_mb = os.path.getsize(QUANT_PATH) / 1024 / 1024
print(f'FP32:  {orig_mb:.1f} MB')
print(f'INT8:  {quant_mb:.1f} MB  ({(1 - quant_mb/orig_mb)*100:.0f}% smaller)')

In [ ]:
# Compare all ONNX variants: size + GPU speed
all_onnx = sorted(glob.glob('/content/models/*.onnx'))
print(f'  {"Model":<35} {"Size MB":>8}  {"avg ms":>8}  {"FPS":>6}')
print('  ' + '-'*60)

precision_data = {}
for mf in all_onnx:
    name = Path(mf).name
    mb   = os.path.getsize(mf) / 1024 / 1024
    try:
        _prov = providers
        _sess = ort.InferenceSession(mf, providers=_prov)
        _inp  = _sess.get_inputs()[0]
        _x    = np.random.randn(1, 3, H, W).astype(np.float32)
        for _ in range(5): _sess.run(None, {_inp.name: _x})
        t0 = time.perf_counter()
        for _ in range(50): _sess.run(None, {_inp.name: _x})
        avg_ms = (time.perf_counter() - t0) * 1000 / 50
        precision_data[name] = {'mb': mb, 'avg': avg_ms}
        print(f'  {name:<35} {mb:>8.1f}  {avg_ms:>8.1f}  {1000/avg_ms:>6.0f}')
    except Exception as e:
        print(f'  {name:<35}  ERROR: {str(e)[:40]}')

---
## 5. TensorRT FP16 — Maximum GPU Throughput

TensorRT is NVIDIA's inference optimizer. It reads ONNX, fuses layers, and compiles a
GPU-specific engine. This is what Autoware uses in production for its perception nodes.

In [ ]:
trt_available = False
try:
    import tensorrt as trt
    import pycuda.driver as cuda
    import pycuda.autoinit
    trt_available = True
    TRT_MAJOR = int(trt.__version__.split('.')[0])
    print(f'TensorRT {trt.__version__}  (major={TRT_MAJOR})')
    print(f'GPU: {torch.cuda.get_device_name(0)}')
except ImportError as e:
    print(f'⚠ TensorRT not available: {e}')

In [ ]:
if trt_available:
    TRT_LOGGER = trt.Logger(trt.Logger.WARNING)

    def build_trt_engine(onnx_path, fp16=True):
        builder = trt.Builder(TRT_LOGGER)
        network = (
            builder.create_network()
            if TRT_MAJOR >= 10 else
            builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
        )
        parser = trt.OnnxParser(network, TRT_LOGGER)
        with open(onnx_path, 'rb') as f:
            if not parser.parse(f.read()):
                for i in range(parser.num_errors):
                    print(f'  Parse error: {parser.get_error(i)}')
                return None

        config = builder.create_builder_config()
        config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 2 << 30)
        if fp16 and builder.platform_has_fast_fp16:
            config.set_flag(trt.BuilderFlag.FP16)
            print('✓ FP16 enabled')

        # Optimization profile (required for any dynamic dims)
        if network.num_inputs > 0:
            inp  = network.get_input(0)
            shp  = tuple(d if d > 0 else 1 for d in inp.shape)
            prof = builder.create_optimization_profile()
            prof.set_shape(inp.name, shp, shp, shp)
            config.add_optimization_profile(prof)

        print('Building TRT engine (may take a few minutes)...')
        serialized = builder.build_serialized_network(network, config)
        if serialized is None:
            print('FP16 failed, retrying FP32...')
            config.clear_flag(trt.BuilderFlag.FP16)
            serialized = builder.build_serialized_network(network, config)
        if serialized is None:
            print('✗ Engine build failed'); return None
        engine = trt.Runtime(TRT_LOGGER).deserialize_cuda_engine(serialized)
        print('✓ Engine ready'); return engine

    engine = build_trt_engine(ONNX_EXPORT_PATH, fp16=True)
else:
    engine = None

In [ ]:
if trt_available and engine is not None:
    class TRTInference:
        """Version-aware TensorRT inference wrapper (TRT 8 / 9 / 10)."""
        def __init__(self, engine):
            self.ctx       = engine.create_execution_context()
            self.major     = int(trt.__version__.split('.')[0])
            if self.major >= 10:
                in_shp  = tuple(engine.get_tensor_shape('image'))
                out_shp = tuple(engine.get_tensor_shape('segmentation'))
            else:
                in_shp  = tuple(engine.get_binding_shape(0))
                out_shp = tuple(engine.get_binding_shape(1))
            self.d_in   = cuda.mem_alloc(int(np.prod(in_shp))  * 4)
            self.d_out  = cuda.mem_alloc(int(np.prod(out_shp)) * 4)
            self.out_shp = out_shp
            self.stream  = cuda.Stream()

        def infer(self, x):
            inp = np.ascontiguousarray(x, dtype=np.float32)
            out = np.empty(self.out_shp, dtype=np.float32)
            cuda.memcpy_htod_async(self.d_in, inp, self.stream)
            if self.major >= 10:
                self.ctx.set_tensor_address('image',        int(self.d_in))
                self.ctx.set_tensor_address('segmentation', int(self.d_out))
                self.ctx.execute_async_v3(stream_handle=self.stream.handle)
            else:
                self.ctx.execute_async_v2([int(self.d_in), int(self.d_out)],
                                          stream_handle=self.stream.handle)
            cuda.memcpy_dtoh_async(out, self.d_out, self.stream)
            self.stream.synchronize()
            return out

    trt_infer   = TRTInference(engine)
    trt_out     = trt_infer.infer(img_np)
    trt_overlay = scene_vis(cv2.resize(frame_bgr, (W, H)), trt_out)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].imshow(cv2.cvtColor(ort_overlay, cv2.COLOR_BGR2RGB)); axes[0].set_title('ONNX Runtime GPU'); axes[0].axis('off')
    axes[1].imshow(cv2.cvtColor(trt_overlay, cv2.COLOR_BGR2RGB)); axes[1].set_title('TensorRT FP16');    axes[1].axis('off')
    plt.suptitle('ONNX Runtime vs TensorRT Output', fontsize=13)
    plt.tight_layout(); plt.show()

    # ── Test 4: TensorRT GPU ──────────────────────────────────────────────────
    trt_times = benchmark(
        lambda: trt_infer.infer(img_np),
        'Test 4 · TensorRT GPU (FP16)',
        use_cuda=True,
    )
else:
    trt_times = None
    print('Skipping TensorRT (not available on this runtime)')

---
## 6. Profiling & Benchmarking

This is what engineers actually do when evaluating deployment options:

| Tool | What it shows |
|------|--------------|
| **Bar chart + error bars** | Average latency with spread |
| **P50 / P95 markers** | Tail-latency (SLO compliance) |
| **Violin plot** | Full latency distribution — not just averages |
| **Run-by-run trace** | Warm-up, jitter, thermal throttling |
| **CDF** | "What % of frames will meet a 30ms target?" |
| **torch.profiler** | Which GPU kernels take the most time |

In [ ]:
# Gather all test results into lists
test_labels, test_times, test_colors = [], [], []

test_labels.append('PyTorch\nCPU');  test_times.append(pytorch_cpu_times);  test_colors.append('#e74c3c')
if pytorch_gpu_times is not None:
    test_labels.append('PyTorch\nGPU'); test_times.append(pytorch_gpu_times); test_colors.append('#3498db')
test_labels.append('ONNX RT\nGPU');  test_times.append(onnx_gpu_times);     test_colors.append('#2ecc71')
if trt_times is not None:
    test_labels.append('TensorRT\nFP16'); test_times.append(trt_times);      test_colors.append('#f39c12')

avgs = np.array([t.mean() for t in test_times])
stds = np.array([t.std()  for t in test_times])
p50s = np.array([np.percentile(t, 50) for t in test_times])
p95s = np.array([np.percentile(t, 95) for t in test_times])
fps  = 1000 / avgs
spds = avgs[0] / avgs
x    = np.arange(len(test_labels))
print(f'Results collected for {len(test_labels)} tests')

In [ ]:
# ── Plot 1: Latency · FPS · Speedup ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# — Latency bar with std error bars + P95 markers
axes[0].bar(x, avgs, yerr=stds, capsize=6, color=test_colors, alpha=0.85, edgecolor='black', linewidth=0.7)
for i, (avg, std, p95) in enumerate(zip(avgs, stds, p95s)):
    axes[0].plot(i, p95, 'v', color='black', markersize=8, zorder=5)
    axes[0].text(i, avg + std + 0.5, f'{avg:.1f}ms', ha='center', va='bottom', fontsize=8, fontweight='bold')
axes[0].set_xticks(x); axes[0].set_xticklabels(test_labels)
axes[0].set_ylabel('Latency (ms)'); axes[0].set_title('Inference Latency\n(bar = avg ± std,  ▼ = P95)')
axes[0].legend([plt.Line2D([0],[0], marker='v', color='black', linestyle='none')], ['P95'], loc='upper right')

# — FPS
axes[1].bar(x, fps, color=test_colors, alpha=0.85, edgecolor='black', linewidth=0.7)
for i, f in enumerate(fps):
    axes[1].text(i, f + 0.3, f'{f:.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[1].set_xticks(x); axes[1].set_xticklabels(test_labels)
axes[1].set_ylabel('Frames per Second'); axes[1].set_title('Throughput  (higher = better)')

# — Speedup
axes[2].bar(x, spds, color=test_colors, alpha=0.85, edgecolor='black', linewidth=0.7)
axes[2].axhline(1.0, color='gray', linestyle='--', linewidth=1)
for i, s in enumerate(spds):
    axes[2].text(i, s + 0.05, f'{s:.1f}x', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[2].set_xticks(x); axes[2].set_xticklabels(test_labels)
axes[2].set_ylabel('Speedup vs PyTorch CPU'); axes[2].set_title('Speedup  (higher = better)')

plt.suptitle('SceneSeg Deployment Benchmark', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Plot 2: Latency Distribution — Violin Plot ───────────────────────────────
# Violin width shows HOW OFTEN each latency value occurs
fig, ax = plt.subplots(figsize=(12, 5))
parts = ax.violinplot(test_times, positions=x, showmedians=True, showextrema=True)

for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(test_colors[i]); pc.set_alpha(0.65)
parts['cmedians'].set_color('black'); parts['cmedians'].set_linewidth(2)

# Scatter individual runs (jittered) to show raw data
for i, times in enumerate(test_times):
    jitter = np.random.uniform(-0.08, 0.08, size=len(times))
    ax.scatter(x[i] + jitter, times, alpha=0.12, s=5, color=test_colors[i])

# P95 markers
for i, p95 in enumerate(p95s):
    ax.hlines(p95, x[i]-0.25, x[i]+0.25, colors='black', linestyles='--', linewidth=1.5, alpha=0.8)
    ax.text(x[i]+0.27, p95, 'P95', fontsize=7, va='center', color='black')

ax.set_xticks(x); ax.set_xticklabels(test_labels)
ax.set_ylabel('Latency (ms)')
ax.set_title('Latency Distribution (100 runs)  —  width = frequency at that latency', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# ── Plot 3: Run-by-Run Latency (GPU tests only) ───────────────────────────────
# Shows warm-up, jitter, thermal throttling
fig, ax = plt.subplots(figsize=(14, 4))
for times, label, color in zip(test_times[1:], test_labels[1:], test_colors[1:]):
    lbl = label.replace('\n', ' ')
    ax.plot(times, alpha=0.75, label=lbl, color=color, linewidth=1.2)
    ax.axhline(times.mean(), color=color, linestyle='--', linewidth=0.8, alpha=0.5)

ax.set_xlabel('Run #'); ax.set_ylabel('Latency (ms)')
ax.set_title('GPU Inference Stability — Run-by-Run  (dashed = mean)', fontsize=12)
ax.legend(loc='upper right'); plt.tight_layout(); plt.show()

In [ ]:
# ── Plot 4: Cumulative Distribution Function ──────────────────────────────────
# Engineers use CDF to answer: "Will we meet a 30ms frame budget 99% of the time?"
fig, ax = plt.subplots(figsize=(12, 5))
for times, label, color in zip(test_times, test_labels, test_colors):
    sorted_t = np.sort(times)
    cdf      = np.arange(1, len(sorted_t) + 1) / len(sorted_t) * 100
    ax.plot(sorted_t, cdf, label=label.replace('\n', ' '), color=color, linewidth=2)

ax.axhline(95, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax.axhline(50, color='gray', linestyle=':',  linewidth=1, alpha=0.7)
ax.text(ax.get_xlim()[1]*0.98, 96, 'P95', ha='right', fontsize=9, color='gray')
ax.text(ax.get_xlim()[1]*0.98, 51, 'P50', ha='right', fontsize=9, color='gray')

ax.set_xlabel('Latency (ms)'); ax.set_ylabel('% of runs faster than X ms')
ax.set_title('Latency CDF — What % of frames meet a given latency budget?', fontsize=12)
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# ── torch.profiler — Kernel-level GPU breakdown ───────────────────────────────
if torch.cuda.is_available():
    with torch.profiler.profile(
        activities=[
            torch.profiler.ProfilerActivity.CPU,
            torch.profiler.ProfilerActivity.CUDA,
        ],
        record_shapes=True,
        profile_memory=True,
    ) as prof:
        for _ in range(20):
            with torch.profiler.record_function('SceneSeg'):
                model_gpu(input_gpu)
        torch.cuda.synchronize()

    print('Top GPU kernels (sorted by CUDA time):')
    print(prof.key_averages().table(sort_by='cuda_time_total', row_limit=12))
else:
    print('torch.profiler requires a GPU runtime')

In [ ]:
# ── Final Summary Table ───────────────────────────────────────────────────────
print('\n' + '='*72)
print('  SCENESEG DEPLOYMENT BENCHMARK')
print('='*72)
print(f'  {"Test":<44} {"avg ms":>7}  {"p95 ms":>7}  {"FPS":>6}  {"vs CPU":>7}')
print(f'  {"-"*44} {"-"*7}  {"-"*7}  {"-"*6}  {"-"*7}')
baseline = avgs[0]
for label, avg, p95 in zip(test_labels, avgs, p95s):
    lbl = label.replace("\n", " ")
    print(f'  {lbl:<44} {avg:>7.1f}  {p95:>7.1f}  {1000/avg:>6.0f}  {baseline/avg:>6.1f}x')
print('='*72)

---
## Conclusion

### What you've built:
- ✓ Loaded an **Autoware production model** (SceneSeg) from TorchScript
- ✓ Ran inference on **real Waymo driving frames**
- ✓ Exported to ONNX and deployed with **ONNX Runtime GPU**
- ✓ Compared **FP32 / FP16 / INT8** precision trade-offs
- ✓ Deployed with **TensorRT FP16** for maximum throughput
- ✓ Profiled with **violin plots, CDF, run-by-run stability, torch.profiler**

### What engineers care about:
- **P95 latency**, not just average — the tail determines SLO compliance
- **Stability** — a model that spikes occasionally is worse than a slower steady one
- **FPS at target precision** — FP16 is usually the sweet spot for AV edge deployment

### Further reading:
- [Autoware vision pilot models](https://github.com/autowarefoundation/autoware_vision_pilot)
- [TensorRT developer guide](https://docs.nvidia.com/deeplearning/tensorrt/developer-guide/)
- [ONNX Runtime performance tuning](https://onnxruntime.ai/docs/performance/tune-performance/)

---
*Neural Optimization by [Think Autonomous](https://thinkautonomous.ai)*